##Initial Setup

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import filter
from delta.tables import DeltaTable
import uuid
from datetime import datetime


In [0]:
spark.sql("Create catalog if not exists ekart_dt")
spark.sql("Use catalog ekart_dt")
spark.sql("Create schema if not exists ekart_dt.Bronze")
spark.sql("create schema if not exists ekart_dt.Silver")
spark.sql("create schema if not exists ekart_dt.Gold")

##Table which stores the watermark for each source _table_
It help the pipeline remember   
- Latest timsstamp already processed
- the latest Pk processed at that Timestamp
- how many rows were written in the latest run
### # This is what make the bronze layer load incremental and rerun-safe


In [0]:
spark.sql("""
            CREATE TABLE IF NOT EXISTS 
            ekart_dt.bronze.ingestion_control(
                layer STRING,                   -- Layer Name
                source STRING,                  -- Source System
                source_file_format STRING,      -- Source Type
                table_name STRING,              -- Source Table Name
                ts_col STRING,                  -- Timestamp Column
                pk_col STRING,                  -- Primary Key Column
                last_successful_ts TIMESTAMP,   -- Last Loaded Timestamp
                last_successful_pk BIGINT,      -- Last Loaded PK
                last_run_id STRING,             -- Last Run ID
                rows_written BIGINT,            -- Rows Loaded
                run_status STRING,              -- Success/Failure
                updated_at TIMESTAMP            -- Metadata Update Time
            )
            USING DELTA
            

        """)

# Source table configuration
This cell define which source tables will be loaded into Bronze and which columns should be used as
-  PK
-  Ts/watermark columns
### It also creates unique Bronze_run_id for the current pipeline 
    


In [0]:
table_config = {
    "customer":{"pk_col":"id","ts_col":"updated_at"},
    "orders":{"pk_col":"id","ts_col":"updated_at"},
    "Payment":{"pk_col":"id","ts_col":"updated_at"},
    "order_items":{"pk_col":"id","ts_col":"updated_at"},
    "transactions":{"pk_col":"id","ts_col":"updated_at"}
}

bronze_run_id = str(uuid.uuid4())
print("current_bronze_run_id :",bronze_run_id)

# Helper Function
This cell contains resuable function 
- get_last_successful_watermark() read the last processed watermark from the control table
- upsert_bronze_control updates the control table after a successful Bronze load
### These function keep the main load logic

In [0]:
def get_last_successful_watermark(table_name):
    ctrl = (
        spark.table("ekart_dt.bronze.ingestion_control")
        .filter(
            (f.col("layer") == "bronze")
            & (f.col("table_name") == table_name)
            & (f.col("run_status") == "success")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )
    row = ctrl.collect()
    if not row:
        return None
    else:
        return row[0]["last_successful_ts"], row[0]["last_successful_pk"]


#Bronze incremental load loop

This is the main Bronze logic.

For each table, the notebook:

1. reads the last watermark  
2. reads the source SQL table  
3. filters only **new / changed rows**  
4. adds Bronze audit columns  
5. appends the rows into the Bronze Delta table  
6. updates the control table  

This is the core incremental loading logic.

In [0]:
def upsert_bronze_control(
    table_name,
    ts_col,
    pk_col,
    last_ts,
    last_pk,
    rows_written,
    run_id
):

    control_df = spark.createDataFrame(
        [(
            "bronze",
            None,
            None,
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.utcnow()
        )],
        schema="""
        layer string,
        source string,
        source_file_format string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(
        spark,
        "ekart_dt.bronze.ingestion_control"
    )

    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            "t.table_name = s.table_name and t.layer = s.layer"
        )
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col",
            "last_successful_ts": "s.last_successful_ts",
            "last_successful_pk": "s.last_successful_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


for table_name, cfg in table_config.items():

    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table_name = {
        "customer": "customers",
        "Payment": "payments"
    }.get(table_name, table_name)
    
    source_table = f"ekart_catalog.dbo.{source_table_name}"
    target_table = f"ekart_dt.bronze.{table_name}"

    watermark = get_last_successful_watermark(table_name)
    if watermark is None:
        last_successful_ts, last_successful_pk = None, None
    else:
        last_successful_ts, last_successful_pk = watermark

    print(f"\n=== Processing {table_name} ===")
    print(f"Last successful ts: {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")

    source_df = (
        spark.read.table(source_table)
        .withColumn(ts_col, f.col(ts_col).cast("timestamp"))
    )

    if last_successful_ts is None:
        rows_to_load = source_df

    else:
        rows_to_load = source_df.filter(
            (f.col(ts_col) > f.lit(last_successful_ts)) |
            (
                (f.col(ts_col) == f.lit(last_successful_ts)) &
                (f.col(pk_col).cast("long") > f.lit(int(last_successful_pk)))
            )
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", f.current_timestamp())
        .withColumn("bronze_run_id", f.lit(bronze_run_id))
        .withColumn("bronze_source_table", f.lit(source_table))
    )

    row_count = rows_to_load.count()

    print(f"{table_name} rows_to_load = {row_count}")

    if row_count == 0:

        print(f"No new rows for {table_name}.")

        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            row_count,
            bronze_run_id
        )

        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_ts = (
        rows_to_load
        .agg(f.max(ts_col).alias("max_ts"))
        .collect()[0]["max_ts"]
    )

    max_pk = (
        rows_to_load
        .filter(f.col(ts_col) == f.lit(max_ts))
        .agg(f.max(f.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        row_count,
        bronze_run_id
    )

    print(f"Wrote {row_count} rows to {target_table}")

In [0]:
%sql
select * from ekart_dt.bronze.ingestion_control